# 02 - Embeddings And Similarity Search

An [embedding](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
represents text as a numeric vector. A retrieval system embeds stored
chunks and a question into the same space, then ranks chunks whose vectors
are close to the question vector. This notebook uses an intentionally
transparent local hash embedding so rankings can be inspected without a
network request; production alternatives are discussed throughout.

**By the end, you will be able to:**

- explain dense vectors and cosine similarity;
- implement a deterministic local embedding baseline with [NumPy](https://numpy.org/doc/stable/);
- perform exhaustive top-k semantic retrieval;
- diagnose where lexical features fail to represent meaning.


## From Text To Nearby Points

Similar texts should map to nearby vector locations. In learned models,
semantic relations emerge from training; in this transparent baseline,
normalized token hashing and a small synonym normalization table make the
mechanics visible. It is useful for learning and tests, not a substitute
for a trained embedding model.

![Semantic vector space relationships](assets/02-word-vector-space.jpg)

Source: [Wikimedia Commons - Word vector illustration](https://commons.wikimedia.org/wiki/File:Word_vector_illustration.jpg)

Sparse lexical vectors and dense learned embeddings solve related but not
identical problems. A dense vector assigns a value throughout a fixed-size
representation, as the Qdrant illustration below shows. Learned dense
embeddings can represent paraphrases even when exact words differ; that
capability is why they are commonly used for RAG.

![Sentence represented as a dense vector](assets/02-dense-vector.png)

Source: [Qdrant - What is a Vector Database?](https://qdrant.tech/articles/what-is-a-vector-database/)


## Similarity Search Flow

```mermaid
graph LR
    A["Chunks"] --> B["Embedding model"]
    B --> C["Normalized vectors"]
    Q["Question"] --> B
    C --> D["Cosine scores"]
    D --> E["Top-k chunks with metadata"]
```

Normalizing vectors to unit length turns a dot product into cosine
similarity. This keeps ranking about direction rather than raw vector
magnitude.


In [ ]:
import hashlib
import json
import re
from pathlib import Path

import numpy as np

# Keep the evidence collection fixed while changing only retrieval mechanics.
documents = json.loads((Path("data") / "course_handbook.json").read_text())
print(f"Loaded {len(documents)} compact policy chunks.")

Each compact source policy is used as one initial chunk. Notebook 01 explained why that is a defensible baseline for this corpus.


In [ ]:
chunks = [
    {
        "chunk_id": f"{document['doc_id']}-whole",
        "doc_id": document["doc_id"],
        "title": document["title"],
        "text": document["text"],
    }
    for document in documents
]
print(chunks[0])

## Build A Transparent Embedding Baseline

A hash embedding assigns normalized tokens to stable numeric dimensions.
A small normalization map handles known paraphrases such as `reimbursement`
and `refund`. Unlike Python's process-randomized `hash()`,
`hashlib.sha256` ensures that re-running the notebook produces the same
vector dimensions.


In [ ]:
SYNONYMS = {
    "reimbursement": "refund",
    "reimburse": "refund",
    "charged": "payment",
    "login": "password",
    "signin": "password",
    "phone": "device",
    "throttle": "limit",
    "slow": "timeout",
    "additional": "extension",
    "duplicated": "duplicate",
    "transfer": "payment",
    "money": "payment",
    "returned": "refund",
    "inference": "prediction",
    "lagging": "slow",
    "fails": "slow",
}
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "can",
    "do",
    "for",
    "from",
    "how",
    "i",
    "in",
    "is",
    "my",
    "of",
    "should",
    "the",
    "to",
    "what",
    "when",
    "with",
}

Tokenization now applies that visible vocabulary policy before vector construction. Keeping it separate makes it easy to inspect which semantic shortcuts the baseline is allowed to use.


In [ ]:
def tokens(text: str) -> list[str]:
    """Lowercase words and normalize a few domain paraphrases."""

    # Basic normalization lets obvious paraphrases share a vector dimension.
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [SYNONYMS.get(word, word) for word in words if word not in STOPWORDS]

In [ ]:
class HashEmbeddingModel:
    """Create inspectable dense vectors without downloading a model."""

    def __init__(self, dimensions: int = 2048) -> None:
        self.dimensions = dimensions

    def encode(self, texts: list[str]) -> np.ndarray:
        # One row per text keeps document and query vectors in the same space.
        vectors = np.zeros((len(texts), self.dimensions), dtype="float32")
        for row, text in enumerate(texts):
            for token in tokens(text):
                # A stable hash makes token placement reproducible across runs.
                digest = hashlib.sha256(token.encode()).digest()
                index = int.from_bytes(digest[:4], "little") % self.dimensions
                vectors[row, index] += 1.0
        # Unit-length vectors turn dot product into cosine similarity.
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        return vectors / np.maximum(norms, 1e-12)

In production, this interface could be replaced with
[Sentence Transformers](https://sbert.net/) or a hosted embedding API
without changing the retrieval contract: `encode(texts)` still returns
vectors whose ranking should be evaluated.


In [ ]:
# Use enough dimensions that classroom results are not driven by hash collisions.
embedder = HashEmbeddingModel(dimensions=2048)
chunk_vectors = embedder.encode([chunk["text"] for chunk in chunks])
print("Vector matrix shape:", chunk_vectors.shape)
print("First vector norm:", round(float(np.linalg.norm(chunk_vectors[0])), 4))

## Cosine Similarity

Cosine similarity measures the angle between vectors. Once vectors are
normalized, `query_vector @ document_vector` is the cosine score:

$$\operatorname{cosine}(q, d) = \frac{q \cdot d}{\lVert q \rVert \lVert d \rVert}$$

![Cosine similarity geometry](assets/02-cosine-similarity.png)

Source: [Wikimedia Commons - Cosine Similarity](https://commons.wikimedia.org/wiki/File:CosineSimilarity.png)


In [ ]:
def search(query: str, top_k: int = 3) -> list[dict[str, object]]:
    """Rank every stored chunk by normalized-vector dot product."""

    query_vector = embedder.encode([query])[0]
    # Exact search scores the query against every stored chunk vector.
    scores = chunk_vectors @ query_vector
    ranked_ids = np.argsort(scores)[::-1][:top_k]
    return [
        {**chunks[index], "score": round(float(scores[index]), 4)}
        for index in ranked_ids
    ]

In [ ]:
query = "I paid twice and need reimbursement for the duplicate invoice."
for result in search(query):
    print(result["doc_id"], result["score"], "-", result["title"])

The expected `refunds` policy should rank first because `reimbursement`
maps to `refund` and the question contains payment/invoice vocabulary. A
rank is evidence about this vectorizer and this corpus; it is not proof
that the generated answer will be correct.


## Inspect Several User Questions

Retrieval design should be judged across distinct intents rather than one convenient query.


In [ ]:
questions = [
    "How do I replace my multi-factor authentication after changing phone?",
    "The API returned 429; when should my client retry?",
    "Can I paste a user's email address into an external model prompt?",
    "How can I request additional time for my project?",
]

for question in questions:
    winner = search(question, top_k=1)[0]
    print(question, "=>", winner["doc_id"], winner["score"])

## Failure Analysis: Similarity Is Not Understanding

Hash features miss conceptual links unless words overlap or we manually
add normalization. Learned dense embedding models usually handle more
paraphrases, but they can still retrieve irrelevant text, encode bias, or
fail on domain-specific terminology. Evaluation remains mandatory.


In [ ]:
hard_questions = [
    "Someone duplicated a transfer; can money be returned?",
    "My second factor is inaccessible after changing hardware.",
    "The inference service is lagging until the request fails.",
    "The scoring endpoint hangs before returning anything.",
]

for question in hard_questions:
    results = search(question, top_k=2)
    print("\n", question)
    for result in results:
        print(" ", result["doc_id"], result["score"])

The first paraphrases are recoverable with the deliberately visible
normalization map. The scoring-endpoint phrasing remains a useful miss: it
shares the intended meaning without sufficient visible vocabulary. A low
or incorrect rank motivates a trained embedding model or additional
evaluation coverage. Do not expand synonym rules endlessly and mistake a
hand-tuned test fixture for robust semantic retrieval.


## Scores Need A Confidence Policy

A top result is always returned, even for a question the corpus does not
answer. Applications should inspect score strength and the margin between
the first and second result rather than turning every nearest neighbor into
an answer. Thresholds are corpus- and embedding-specific; they must be
tuned with supported and unsupported examples.


In [ ]:
confidence_questions = [
    "How should I reset a forgotten password?",
    "Where can I park a bicycle near the campus?",
    "What recipe should I cook tonight?",
]

for question in confidence_questions:
    first, second = search(question, top_k=2)
    # A narrow top-two margin signals uncertainty even when one item ranks first.
    margin = float(str(first["score"])) - float(str(second["score"]))
    print(question)
    print(
        " top=", first["doc_id"], "score=", first["score"], "margin=", round(margin, 4)
    )

Unsupported questions may still overlap accidentally with a source or tie
at zero. A robust RAG pipeline combines retrieval thresholds, a prompt that
permits `I do not know`, and answer-grounding evaluation rather than using
the top score as truth.


## Metadata Enables Better Retrieval Products

Vector similarity is only one part of retrieval. Metadata lets an
application filter by policy section, cite the original page, implement
access rules, and remove outdated records. The helper below demonstrates a
simple section filter after ranking; vector databases can often perform
such filtering inside the search request.


In [ ]:
def search_in_section(
    query: str, section: str, top_k: int = 2
) -> list[dict[str, object]]:
    """Return scored results only from an allowed handbook section."""

    all_scores = chunk_vectors @ embedder.encode([query])[0]
    # Filter after scoring here for clarity; databases may filter during search.
    allowed = [
        (index, float(score))
        for index, score in enumerate(all_scores)
        if documents[index]["section"] == section
    ]
    allowed.sort(key=lambda pair: pair[1], reverse=True)
    return [
        {**chunks[index], "score": round(score, 4)} for index, score in allowed[:top_k]
    ]


for result in search_in_section("I paid twice for one invoice.", "Billing"):
    print(result["doc_id"], result["score"], result["title"])

In a real service, filtering after retrieving only a few candidates can
incorrectly remove all useful results. Apply filters during retrieval or
retrieve a sufficiently broad candidate pool, then evaluate the combined
policy.


## Exercise - Retrieve A Support Answer Source

Use `search()` to retrieve two sources for:

`My password link expired before I could open it.`

Check whether `password-reset` appears at rank 1 and print the supporting
text that would be passed to a generator.


## Correction


In [ ]:
exercise_query = "My password link expired before I could open it."
exercise_results = search(exercise_query, top_k=2)
# Rank-one correctness is the simplest useful check for this known question.
correct_at_one = exercise_results[0]["doc_id"] == "password-reset"

print("Relevant result at rank 1:", correct_at_one)
for rank, result in enumerate(exercise_results, start=1):
    print(f"Rank {rank}: {result['doc_id']} ({result['score']})")
    print(result["text"])

**Reference answer:** `password-reset` should appear first because it
contains both the reset-link action and its expiry condition. Supplying
that full chunk to generation preserves the instruction to request a new
link rather than inventing a recovery process.


## Next Step And Further Reading

This notebook scores every document vector. That exact scan is transparent
but expensive at large scale. Next, FAISS will provide optimized exact and
approximate index structures while retaining chunk metadata outside the
vector store.

- [Sentence Transformers semantic search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
- [NumPy linear algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)
- Reimers and Gurevych, [Sentence-BERT](https://arxiv.org/abs/1908.10084)
- [OpenAI embeddings guide](https://platform.openai.com/docs/guides/embeddings)
